In [1]:
import tensorflow as tf
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Flatten, Dense, Dropout
from tensorflow.keras.regularizers import l1, l2

# Load CIFAR-10
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalize
x_train = x_train / 255.0
x_test = x_test / 255.0
y_train = y_train.flatten()
y_test = y_test.flatten()

# Model configurations
configs = [
    ('Baseline', None, None),
    ('Xavier', 'glorot_uniform', None),
    ('Kaiming', 'he_normal', None),
    ('Dropout', None, 'dropout'),
    ('L1', None, 'l1'),
    ('L2', None, 'l2')
]

optimizers = ['adam', 'sgd']
results = []

# Train 12 models
for name, init, reg in configs:
    for opt in optimizers:

        model = Sequential([
            Input(shape=(32, 32, 3)),
            Flatten()
        ])

        for units in [256, 128]:
            kwargs = {'activation': 'relu'}

            if init:
                kwargs['kernel_initializer'] = init

            if reg == 'l1':
                kwargs['kernel_regularizer'] = l1(0.001)
            elif reg == 'l2':
                kwargs['kernel_regularizer'] = l2(0.001)

            model.add(Dense(units, **kwargs))

            if reg == 'dropout':
                model.add(Dropout(0.5))

        model.add(Dense(10, activation='softmax'))

        model.compile(
            optimizer=opt,
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )

        print(f"\nTraining {name} with {opt}")

        model.fit(
            x_train, y_train,
            epochs=5,
            batch_size=128,
            verbose=1
        )

        loss, acc = model.evaluate(x_test, y_test, verbose=0)

        results.append([
            name, reg if reg else 'None',
            opt.upper(), round(acc * 100, 2),
            round(loss, 4)
        ])

# Comparison table
df = pd.DataFrame(results, columns=[
    'Model', 'Regularization', 'Optimizer',
    'Test Accuracy (%)', 'Test Loss'
])

print("\nMODEL PERFORMANCE COMPARISON")
print(df.to_string(index=False))


Training Baseline with adam
Epoch 1/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 9s 18ms/step - accuracy: 0.3210 - loss: 1.8953
Epoch 2/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.3943 - loss: 1.6995
Epoch 3/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.4252 - loss: 1.6156
Epoch 4/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - accuracy: 0.4426 - loss: 1.5632
Epoch 5/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - accuracy: 0.4613 - loss: 1.5182

Training Baseline with sgd
Epoch 1/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.2858 - loss: 1.9998
Epoch 2/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.3591 - loss: 1.8203
Epoch 3/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.3854 - loss: 1.7520
Epoch 4/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.4043 - loss: 1.6997
Epoch 5/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.4165 - loss: 1.6629

Training Xavier with adam
Epoch 1/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step -